# Python Basics - coding challenges

10 hands-on problems. Each solution cell ends with an assertion so you can check yourself.

## Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

pandas 3.0.5


# Welcome: 10 Coding Challenges

This notebook is the sharpest tool in your Week 1 interview-prep kit: **ten hands-on problems** with solutions already in place, each ending in an assertion you can run to verify your own attempt. The rule of engagement is simple — write your version of the solution in the scratch area first, test it against the assertion, and only then compare with the provided answer. (A `## Setup` cell above has already imported `numpy` and `pandas` and seeded a random number generator for your experiments.)

Before you start, here is the honest map of what these ten challenges exercise, and where each skill came from in the earlier sessions:

- **Digits and maths (C1, C4)** — digits in reverse as a list, and the classic FizzBuzz. C1 revisits the `% 10` / `// 10` digit extraction you first met in Session 2's pyramid and series work; C4 tests your command of the ternary expression chain from Session 3.
- **Dictionaries and counting (C2, C6)** — word-frequency counts and running maxima. Both are accumulator problems: C2 counts with `dict.get(key, 0) + 1`, C6 keeps a 'best so far' running through the values.
- **Strings and text (C3, C5, C9)** — the longest word, anagram checking, and run-length compression. C5 introduces the tally-dictionary pattern that swaps plus-one for minus-one as you walk the second input; C9 is a state machine walking characters and counting runs.
- **Lists and structure (C7, C8)** — flattening one level of nesting with a nested list comprehension, and finding sum-pairs in a single pass using a `set` for O(1) lookups.
- **Stacks (C10)** — balanced brackets, the very first 'use a stack' problem you will meet.

Each theory cell before a challenge explains the idea so you can attempt it independently; every demonstration cell keeps its usual meaning. Time yourself — the notebook's own closing note suggests ten minutes per challenge as a realistic screening pace. Solve confidently, verify with the assertion, then read the reference solution and note any trick it used that you missed. Happy solving!

## Challenge 1

Write a function that returns the digits of an integer in reverse order as a list.

## Challenge 1: Digits of an integer in reverse

Your goal: given an integer — the sample is `1230` — return its digits, in reverse order, as a list: `[0, 3, 2, 1]`.

Think about what has to happen step by step:

1. **Normalise negativity.** `1230` is positive, but `-1230` would be a hostile input. Using `abs(n)` keeps the digit logic pure — no minus signs in the list.
2. **Make the digits addressable.** `str(n)` turns the number into the string `'1230'`, and strings support indexing and iteration — you can loop over their characters.
3. **Convert each character to an integer.** A list comprehension `[int(d) for d in ...]` walks the string and maps every `'1'` to `1`. This is the comprehension pattern from Session 2, now applied to text.
4. **Reverse the order.** The slice `[::-1]` flips the list so the last digit comes first — giving `[0, 3, 2, 1]` for `1230`.

The reference solution is that whole story in one line:

```
[int(d) for d in str(abs(n))][::-1]
```

Two ideas worth filing away:

- **Read it inside-out:** the comprehension produces the digits left-to-right, and only then is the whole list reversed. Order of operations inside an expression mirrors the order of reasoning.
- **The arithmetic alternative:** purely with maths you would do `n % 10` (grab last digit) then `n // 10` (chop it off) in a loop, collecting into a list — the digit-extraction ladder from Session 2. String conversion is cleaner here; the `%`/`//` version is what you would use to reverse a number *without* touching strings. Both are worth knowing for the interview.

In [2]:
def reversed_digits(n):
    return [int(d) for d in str(abs(n))][::-1]

assert reversed_digits(1230) == [0, 3, 2, 1]
print('ok')

ok


## Challenge 2

Count word frequency in a sentence without importing collections.

## Challenge 2: Word frequency without collections

The task: count how many times each word appears in a sentence — `'a b a'` should map `'a'` to `2` and `'b'` to `1` — and you may not use the `collections` module's `Counter`.

The dictionary is your tool, and the whole algorithm is an accumulator with a single trick line:

```
out[word] = out.get(word, 0) + 1
```

Let's decode that line, because it is the single most reused one-line pattern in all of Python counting:

- `out.get(word, 0)` looks up `word` in the dictionary; if it is missing, `0` is returned instead of raising `KeyError`.
- `+ 1` increases that count.
- The assignment stores the new count back under `word`. First time you see `'a'`: `get` gives `0`, store `1`. Second time: `get` gives `1`, store `2`.

Before that loop runs, the sentence must be prepared, and this is where text nuances hide:

1. **Lowercase everything** — `.lower()` so `'The'` and `'the'` count as the same word. The sample sentence `'a b a'` has no capitals, but real text does.
2. **Split into words** — `.split()` cuts on whitespace, producing the list `['a', 'b', 'a']`.
3. **Aggregate in a loop** — one `for` over that list with the one-liner above.

This is the dictionary-accumulator pattern from the Session 3 string problems, formalised. It also hints at exactly what `collections.Counter` does internally — so if you later switch to it, you will know what it is doing under the hood and can explain that in the interview. Frequency counting of words, letters, IDs, and errors is a daily data-cleaning task, and `get(key, default)` is its engine.

In [3]:
def freq(text):
    out = {}
    for w in text.lower().split():
        out[w] = out.get(w, 0) + 1
    return out

assert freq('a b a')['a'] == 2
print('ok')

ok


## Challenge 3

Return the longest word in a sentence, ties broken by first occurrence.

## Challenge 3: The longest word

Find the word with the greatest length in a sentence, with any ties going to the **first** occurrence — `'data science is powerful'` should give `'powerful'`.

Python hands you the answer in one line through the `max` function's *key* argument:

```
max(text.split(), key=len)
```

Decompose it:

- `text.split()` produces the word list: `['data', 'science', 'is', 'powerful']`.
- `max(..., key=len)` scans that list and, instead of comparing the words themselves (which would compare alphabetically), compares `len(word)` for each — and returns the *word*, not the length.

The `key` parameter is the gateway to a whole family of 'find the element that is best by some measure' problems: `max(items, key=len)` for longest, `min(items, key=len)` for shortest, and it works on any list of objects as long as you supply a function that maps each one to a number that can be compared.

Three finer points worth understanding:

- **Ties go to the first occurrence** because `max` — like `min` — keeps the first element when several are equal under the key. That is exactly the behaviour the challenge requires, for free.
- **The key function is applied to each element once** conceptually; you are free to pass any one-argument function, from the built-in `len` to a `lambda` or even a method.
- **Empty input** would make `max` raise `ValueError`; a well-guarded version checks the list is non-empty first (the reference doesn't, but you can note the edge case aloud in an interview).

This single keyword argument — `key=` — appears again and again after Week 1: sorting by attribute, grouping by value, and ranking all lean on it. Master it here with one line and you save yourself hundreds of lines later.

In [4]:
def longest(text):
    return max(text.split(), key=len)

assert longest('data science is powerful') == 'powerful'
print('ok')

ok


## Challenge 4

Implement FizzBuzz returning a list for n = 15.

## Challenge 4: FizzBuzz for numbers 1 to n

FizzBuzz is the most famous screening problem in programming, and the version here asks for numbers `1..n` turned into a list of strings: multiples of both 3 and 5 become `'FizzBuzz'`, multiples of 3 become `'Fizz'`, multiples of 5 become `'Buzz'`, and everything else stays a number string.

The reference answer is a comprehension whose expression is a **chain of ternary conditionals**, and the order of those conditions is the entire game:

```
['FizzBuzz' if i % 15 == 0 else 'Fizz' if i % 3 == 0 else 'Buzz' if i % 5 == 0 else str(i)
 for i in range(1, n + 1)]
```

Why does the order matter so much? Because a ternary chain returns the *first* branch whose condition is true. If you checked `i % 3 == 0` first, 15 would be classified as `'Fizz'` — since 15 is divisible by 3 too — and the 'FizzBuzz' branch would never fire. The two conditions must be tested in the right sequence:

1. `i % 15 == 0` — divisible by both (equivalently, by LCM 15) — you can also write `i % 3 == 0 and i % 5 == 0`.
2. else `i % 3 == 0` — pure Fizz.
3. else `i % 5 == 0` — pure Buzz.
4. else the number itself, via `str(i)`.

The conditional is *right-associative*: `a if c1 else b if c2 else d` reads as `a if c1 else (b if c2 else d)` — each `else` opens the next mini-decision.

Two takeaways beyond the joke problem:

- **The compound condition test** `% 15` vs the pairwise `and` — knowing both lets you explain the choice under questioning.
- **The `if...else` expression inside a comprehension** expands your one-liner vocabulary significantly: the same shape solves formatting, mapping, and normalisation problems.

In [5]:
def fb(n):
    return ['FizzBuzz' if i % 15 == 0 else 'Fizz' if i % 3 == 0 else 'Buzz' if i % 5 == 0 else str(i) for i in range(1, n + 1)]

assert fb(15)[-1] == 'FizzBuzz'
print('ok')

ok


## Challenge 5

Check whether two strings are anagrams in O(n) time.

## Challenge 5: Anagrams in O(n) time

The definition: two strings are anagrams when they use exactly the same letters the same number of times — `'listen'` and `'silent'` are anagrams; `'cat'` and `'catt'` are not.

The reference guarantees O(n) — proportional to the length of the input — which rules out the naive `sorted(a) == sorted(b)` (that is O(n log n)). The linear-time trick is the **tally dictionary**, and it works in one pass per string:

1. **Early exit on length.** If `len(a) != len(b)`, they cannot use the same letter counts — return `False` immediately. This is your cheap rejection test.
2. **Count up through the first string.** For each character of `a`, increment its tally: `tally[ch] = tally.get(ch, 0) + 1` — the accumulator from Challenge 2.
3. **Count down through the second string.** For each character of `b`, decrement its tally. If any tally dips below zero, the strings do not match — `b` has a letter `a` does not have, or has too many of it — return `False` now.
4. **Fall through to `True`.** If every decrement stays non-negative and the lengths matched, the tallies would have to be perfectly offset — the strings are anagrams.

Why is the final `return True` safe? Lengths equal, all counts balanced, nothing went negative — by the pigeonhole principle the tallies end exactly at zero.

Two ideas this challenge plants for later:

- **Early exits** chop average runtime dramatically — length checks, negativity checks — and interviewers love seeing them because they show you think about waste.
- **The candidate-rejection trick** (fail on the first inconsistency) generalises to many 'compare two collections' problems.

Run the assertion, then try `'evil'`/`'vile'` (True) and `'abb'`/`'aab'` (False) to feel the negative-tally exit fire.

In [6]:
def anagram(a, b):
    if len(a) != len(b):
        return False
    tally = {}
    for ch in a:
        tally[ch] = tally.get(ch, 0) + 1
    for ch in b:
        tally[ch] = tally.get(ch, 0) - 1
        if tally[ch] < 0:
            return False
    return True

assert anagram('listen', 'silent')
print('ok')

ok


## Challenge 6

Compute the running maximum of a list.

## Challenge 6: Running maximum

Given a list like `[2, 5, 3, 9]`, output, at every position, the largest value seen so far: `[2, 5, 5, 9]`. This is the running (or prefix) maximum, and it is a one-pass sweep with two pieces of state.

Setting up the sweep:

- **A 'best so far' accumulator** — start it at the smallest possible value. The reference uses `float('-inf')`, which is smaller than any real number, so the first value always wins the comparison. (Starting from the first element is the other common choice.)
- **A result list** — append the fresh best at every step.

The loop body compresses the whole idea into two tiny lines:

```
best = max(best, v)
out.append(best)
```

Walk it on `[2, 5, 3, 9]`: v=2 → best=2 → out `[2]`; v=5 → best=5 → `[2,5]`; v=3 → best stays 5 → `[2,5,5]`; v=9 → best=9 → `[2,5,5,9]`. The value 3 never changes `best` because 5 is already greater — exactly the sense in which it is a *running* maximum: the history of the best.

Three ideas to keep:

- **The accumulator + loop-append rhythm** is the same shape you used for sum and series problems in Session 2 — only the update rule differs (max instead of sum).
- **`max(best, v)` vs a hand-written `if`** — the built-in keeps the code declarative; write the `if` version too so you can claim both styles.
- **`float('-inf')` is a deliberate sentinel** — it is the identity for `max`, making the first comparison trivially correct. Choosing the right sentinel is itself an interview talking point.

Running aggregates — max, min, sum, count — turn a list into its own history, and that perspective unlocks streaming and time-series processing later in the course.

In [7]:
def running_max(values):
    out, best = [], float('-inf')
    for v in values:
        best = max(best, v)
        out.append(best)
    return out

assert running_max([2, 5, 3, 9]) == [2, 5, 5, 9]
print('ok')

ok


## Challenge 7

Flatten a list of lists one level deep without itertools.

## Challenge 7: Flatten a list of lists, one level deep

You are handed nested lists like `[[1, 2], [3]]` and must produce the single flat list `[1, 2, 3]` — without `itertools.chain` or `chain.from_iterable`.

The one-liner is a **nested list comprehension**, and reading its order correctly is the entire skill:

```
[x for sub in nested for x in sub]
```

Comprehension clauses read **left to right, then nested**: the outer `for sub in nested` runs first, giving `sub = [1, 2]`, then the inner `for x in sub` iterates its elements `1`, `2`; then control returns to the outer loop for `sub = [3]` and yields `3`. The expression `x` is what lands in the output list.

So the mental model:

- `for sub in nested` selects each inner list.
- `for x in sub` unpacks it.
- The output collects every `x` in that order.

Why exactly one level deep? Because each `sub` is iterated as a list of elements, but those elements are never themselves flattened — if a `sub` contained another list (three levels of nesting), that inner list would simply be copied in as a single element. That is why the challenge says 'one level deep'.

Three notes worth keeping:

- **Comprehension order is the opposite of the enclosing `for` in block code** — in a regular loop you would write the outer loop first and nest the inner one; in the comprehension, the expression comes first and the two `for`s follow in loop order. Confusing at first, mechanical once practised.
- **Alternatives exist for a reason**: `sum(nested, [])` also flattens but is an accidental quadratic; itertools does it cleanly in production. Knowing your manual one-liner means you understand what those tools do internally.
- This same nesting order powers matrix transposition and any 'expand one collection of collections' job.

In [8]:
def flatten(nested):
    return [x for sub in nested for x in sub]

assert flatten([[1, 2], [3]]) == [1, 2, 3]
print('ok')

ok


## Challenge 8

Find all pairs in a list summing to a target, using a single pass.

## Challenge 8: Pairs that sum to a target, in one pass

Given `[1, 3, 2, 4]` and a target `5`, list the pairs that add up to it — the reference returns `[(3, 2), (1, 4)]`. The restriction is the interesting part: a **single pass** over the input.

The insight that makes single-pass possible is arithmetic, not scanning: for the current value `v`, the partner we need already held in memory would be `target - v`. So we keep a `set` of everything seen so far, and for each new value ask: *have I already met its complement?*

```
seen = set()
for v in values:
    if target - v in seen:
        out.append((target - v, v))
    seen.add(v)
```

Walk `[1, 3, 2, 4]` with target 5:

- v=1: complement 4 not in `seen {}` → no pair; add 1.
- v=3: complement 2 not in `{1}` → no pair; add 3.
- v=2: complement 3 **is** in `{1, 3}` → record `(3, 2)`; add 2.
- v=4: complement 1 **is** in `{1, 3, 2}` → record `(1, 4)`; add 4.

Result: `[(3, 2), (1, 4)]`. Note each pair is discovered *exactly once*, at the moment its second member arrives — nothing is double-counted.

Three structural lessons:

- **Sets give O(1) membership tests**, which is what saves the nested loop; the time is O(n), the space O(n). Trade memory for speed — a recurring, interview-worthy decision.
- **`seen.add(v)` runs unconditionally after the check** — crucially *after*, so a value never pairs with itself unless the target genuinely demands it. The placement of this line is the classic bug in this problem.
- **Complement programming** — 'what must I have seen already to complete this value?' — is a template for two-sum, difference, and many other pair/triple problems.

In [9]:
def pairs(values, target):
    seen, out = set(), []
    for v in values:
        if target - v in seen:
            out.append((target - v, v))
        seen.add(v)
    return out

assert pairs([1, 3, 2, 4], 5) == [(3, 2), (1, 4)]
print('ok')

ok


## Challenge 9

Compress a string: 'aaabb' becomes 'a3b2'.

## Challenge 9: Run-length string compression

The goal: collapse runs of repeated characters into compact form, so `'aaabb'` becomes `'a3b2'`. This is run-length encoding — a real compression scheme used in bitmap images and fax data.

The way to think about it is as a tiny **state machine**: you walk the string holding 'the character of the current run' and 'how long the run is so far', and you act on *change*. When the current character equals the run character, you extend the run; the moment it differs, you flush the finished run to the output and start a new one.

Stepping through the reference logic:

1. **Guard the empty string** — `if not s: return ''`. Easy to forget, deadly if forgotten.
2. **Seed the state** — the first character becomes `prev`, and the run length `n` starts at 0 (the loop below increments it on first contact).
3. **Walk with one branch**: same character → `n += 1`; different character → append `f'{prev}{n}'` to the output, reset `prev` and `n` to the new character and 1.
4. **Empty the buffer** — after the loop ends, one run is always still open; flush it with a final append, then `''.join(out)` stitches the pieces.

For `'aaabb'`: the `a` run collects 3 (`a3`), the `b` run collects 2 (`b2`), final join gives `a3b2`.

Three ideas to bank:

- **The 'flush on change + final flush' discipline** is the eternal bug in state-machine code — the last run never fires its own 'change' event, so it must be flushed manually.
- **`prev` and `n` are a compact pair of state variables** representing 'what I'm currently counting'. Naming and managing them cleanly is the difference between this solution and a tangled mess.
- **The f-string `f'{prev}{n}'` is the ordinary f-string formatting from Session 3**, now doing text-assembly inside the loop.

Run-length encoding is your first actual compression algorithm; understanding when runs are long (they compress well) and when they are not (they don't) is the seed of much deeper information theory.

In [10]:
def compress(s):
    if not s:
        return ''
    out, prev, n = [], s[0], 0
    for ch in s:
        if ch == prev:
            n += 1
        else:
            out.append(f'{prev}{n}')
            prev, n = ch, 1
    out.append(f'{prev}{n}')
    return ''.join(out)

assert compress('aaabb') == 'a3b2'
print('ok')

ok


## Challenge 10

Validate balanced brackets in an expression.

## Challenge 10: Balanced brackets

The job: decide whether `'{[()]}'` is balanced and `'(]'` is not — every opening bracket must be closed by the matching one, in the correct nesting order.

The data structure that does exactly this is the **stack**: a last-in, first-out pile. What you need is to remember, at any moment, which *most recent* opening bracket is waiting for its close. The stack is that memory.

```
pairs = {')': '(', ']': '[', '}': '{'}
stack = []
for ch in s:
    if ch in '([{':          # opening → push
        stack.append(ch)
    elif ch in pairs:        # closing → must match top
        if not stack or stack.pop() != pairs[ch]:
            return False
return not stack
```

Walk through the algorithm as a protocol:

1. **Push on open.** `(` `[` `{` all go onto the stack. After `{[(` the stack holds `['{', '[', '(']`.
2. **Pop and verify on close.** A `)` must find `(` on top, else the string is unbalanced. In `'{[()]}'`, the first close is `)` popping `(` — match! Then `]` pops `[`, then `}` pops `{`, stack ends empty.
3. **Two failure modes.** A closer that finds an empty stack (`')('` — closes before anything opens) returns `False`; a closer that meets the wrong opener (`'(]'`, where `]` meets `(`) also returns `False`.
4. **The final `return not stack`** catches 'left-over openers': `'({'` pushes a balanced pair internally but the stack is left with `['(']` after the loop — unconsumed openers mean unbalanced.

The mapping dict `pairs` is the elegance of the solution: it turns 'must be the matching bracket' into a single dictionary lookup.

Why programmers care: compilers, parsers, HTML validators, and every nested-structure checker use exactly this stack protocol. The `(, [, {` classification and the pop-and-compare moment are the two ideas to remember — they generalise to tags, parentheses in expressions, and JSON validation.

In [11]:
def balanced(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in '([{':
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack

assert balanced('{[()]}')
assert not balanced('(]')
print('ok')

ok


## After the session

Time yourself: 10 minutes per challenge is a realistic screening pace. Re-do the ones that took longer.

## Summary

The ten challenges marched you through the exact skill set a Week 1 screening test looks for, and every one of them leaned on patterns you had already practised in Sessions 1-3. Digits: you reversed them via strings and comprehensions (C1) and learned the `%` modulus ladder that drives FizzBuzz (C4). Dictionaries: you counted word frequencies with `dict.get(key, 0) + 1` (C2), rated the longest word through `max(..., key=len)` (C3), and verified anagrams in O(n) with a tally that counts up then down (C5). Aggregates and structure: running maxima with a 'best so far' accumulator (C6), one-level flattening via the nested comprehension with its counter-intuitive reading order (C7), and single-pass sum-pairs achieved by remembering complements in a `set` (C8). Text machinery: run-length compression as a state machine with a mandatory final flush (C9). And the stack: balanced brackets by pushing opens and popping-and-verifying closes (C10), your first genuine data-structure problem.

If you finished all ten within the ten-minutes-each target, you have the computational reflexes interviewers probe on day one. Redo the long ones, and remember — the assertions are your scaffold: whenever you doubt your solution, run the check. Move on to the session recap, and keep this notebook as your warm-up deck for future interview drills. Happy coding!